# Cross-encoder (xlm-roberta-base, MIT) — Google Colab GPU

Reads both records together, `S1 name | address` [SEP] `candidate name | address`, and outputs P(match).
Attention across the pair spots decoy tweaks (house number, extra word, legal form) that hand-made features miss.

Reads everything from and writes everything to Drive (`amazon ml new arch/`) - nothing to download or upload.

**Runtime → Change runtime type → T4 GPU.** Needs on Drive: `er_work/run_frac0.15_seed42/v4/ce_val_pairs.parquet`,
`er_work/test_run_v4/ce_test_pairs.parquet`, and the raw `train/` + `test/` TSVs (or `new_parquet/` copies).

Writes `er_work/ce/`: `ce_val_scores.parquet`, `ce_test_scores.parquet`, `ce_model/`, `ce_report.json`.
After a disconnect: Run all again - a finished model is reloaded instead of retrained, finished score files are kept.

In [1]:
# ===== CELL 1: setup =====
!pip -q install transformers sentencepiece
from google.colab import drive
drive.mount("/content/drive")
import os, glob, json, time, math, gc, csv
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, log_loss

MODEL_NAME = os.environ.get("CE_MODEL", "xlm-roberta-base")   # MIT, 278M params
MAX_LEN    = 96                   # "name | address" pairs are short; 96 tokens covers nearly all
HOLD       = 2                    # must match the validation export cell
EPOCHS     = 2
BATCH      = 64                   # total over GPUs
LR         = 2e-5
MAX_TRAIN_MIN = 150               # stop training after this many minutes (keeps time for scoring)
ROOT       = "/content/drive/MyDrive/amazon ml new arch"
OUT        = f"{ROOT}/er_work/ce"; os.makedirs(OUT, exist_ok=True)

def find(rel):
    pth = f"{ROOT}/{rel}"
    return pth if os.path.exists(pth) else None

VAL_PAIRS  = find("er_work/run_frac0.15_seed42/v4/ce_val_pairs.parquet")
TEST_PAIRS = find("er_work/test_run_v4/ce_test_pairs.parquet")
assert VAL_PAIRS and TEST_PAIRS, "ce_val_pairs.parquet / ce_test_pairs.parquet not found on Drive - run the two export cells first"
dev = "cuda" if torch.cuda.is_available() else "cpu"
NGPU = torch.cuda.device_count()
print("device", dev, "| GPUs", NGPU, "|", VAL_PAIRS, TEST_PAIRS)

Mounted at /content/drive
device cuda | GPUs 1 | /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/v4/ce_val_pairs.parquet /content/drive/MyDrive/amazon ml new arch/er_work/test_run_v4/ce_test_pairs.parquet


In [2]:
# ===== CELL 2: record text (raw name | address) for every id we need =====
READ_KW = dict(sep="\t", dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False, na_filter=False, encoding="utf-8")

def raw_source(split, k):
    names = ([f"source{k}.parquet", f"train_source{k}.parquet"] if split == "train" else [f"test_source{k}.parquet"])
    for n in names:
        pth = find(f"new_parquet/{n}")
        if pth:
            return pd.read_parquet(pth, columns=["entity_id", "business_name", "business_address"])
    pth = find(f"{split}/{split}_source{k}.tsv")
    assert pth, f"raw {split} source {k} not found"
    return pd.read_csv(pth, usecols=["entity_id", "business_name", "business_address"], **READ_KW)

def texts_for(split, ids):
    ids = set(ids); out = {}
    for k in (1, 2, 3):
        d = raw_source(split, k)
        d = d[d.entity_id.isin(ids)]
        out.update(dict(zip(d.entity_id.astype(str), (d.business_name.fillna("").astype(str) + " | " +
                                                      d.business_address.fillna("").astype(str)))))
        del d; gc.collect()
    return out

t0 = time.time()
val = pd.read_parquet(VAL_PAIRS); test = pd.read_parquet(TEST_PAIRS)
TX_tr = texts_for("train", pd.concat([val.s1_id, val.pool_id]).unique())
TX_te = texts_for("test", pd.concat([test.s1_id, test.pool_id]).unique())
for df, tx in ((val, TX_tr), (test, TX_te)):
    df["a"] = df.s1_id.map(tx).fillna(""); df["b"] = df.pool_id.map(tx).fillna("")
print(f"val pairs {len(val):,} | test pairs {len(test):,} | texts loaded in {(time.time() - t0) / 60:.1f} min")
print(val[["a", "b", "y", "p"]].head(5).to_string())

val pairs 761,926 | test pairs 2,191,300 | texts loaded in 2.3 min
                                                                                                                                                     a                                                                                                                                                   b  y         p
0  Dream Construction Limited | H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sagar Hotel Building, Opp.Rta Office, Mo, Osarambagh, Hyderabad, Telangana                                                               డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్ | H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana  1  0.999207
1  Dream Construction Limited | H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sagar Hotel Building, Opp.Rta Office, Mo, Osarambagh, Hyderabad, Telangana          Mr Dream Limited Construction | NO 16-11-23/37/A , 2ND FLOOR, FLAT NO.207, SAGAR HOTEL BUILDING, OPP.RTA OFFICE, MO, OSARAMBAGH, Telangana  1  0.9999

In [3]:
# ===== CELL 3: model + batching helpers =====
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def batches(a, b, idx, bs):
    """Length-sorted batches (fast padding) over row indices idx."""
    lens = np.array([len(a[i]) + len(b[i]) for i in idx])
    order = idx[np.argsort(lens, kind="stable")]
    for s in range(0, len(order), bs):
        yield order[s:s + bs]

def encode(a, b, rows):
    enc = tok([a[i] for i in rows], [b[i] for i in rows], truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors="pt")
    return {k: v.to(dev, non_blocking=True) for k, v in enc.items()}

@torch.no_grad()
def score(model, df, bs=512):
    model.eval()
    a, b = df.a.tolist(), df.b.tolist(); out = np.zeros(len(df), np.float32)
    idx = np.arange(len(df)); t0 = time.time(); done = 0
    for rows in batches(a, b, idx, bs):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            logits = model(**encode(a, b, rows)).logits.squeeze(-1)
        out[rows] = torch.sigmoid(logits.float()).cpu().numpy(); done += len(rows)
        if done % (bs * 400) < bs:
            print(f"  scored {done:,}/{len(df):,} ({time.time() - t0:.0f}s)", flush=True)
    return out

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [4]:
# ===== CELL 4: train on folds != HOLD =====
tr = val[val.fold != HOLD].reset_index(drop=True)
print(f"train rows {len(tr):,} | positives {int(tr.y.sum()):,}")
RESUME = os.path.exists(f"{OUT}/ce_model/config.json")
model = AutoModelForSequenceClassification.from_pretrained(f"{OUT}/ce_model" if RESUME else MODEL_NAME, num_labels=1).to(dev)
net = torch.nn.DataParallel(model) if NGPU > 1 else model
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(tr) / BATCH)
sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
lossf = torch.nn.BCEWithLogitsLoss()
a, b, yy = tr.a.tolist(), tr.b.tolist(), tr.y.values.astype(np.float32)
t0 = time.time(); step = 0; stop = False
if RESUME:
    print('trained model found on Drive - skipping training')
for ep in range(0 if RESUME else EPOCHS):
    net.train()
    perm = np.random.RandomState(ep).permutation(len(tr))
    for s in range(0, len(perm), BATCH):
        rows = perm[s:s + BATCH]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            logits = net(**encode(a, b, rows)).logits.squeeze(-1)
            loss = lossf(logits.float(), torch.tensor(yy[rows], device=dev))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step(); step += 1
        if step % 500 == 0:
            print(f"epoch {ep + 1} step {step:,}/{steps:,} loss {loss.item():.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
        if (time.time() - t0) / 60 > MAX_TRAIN_MIN:
            print("time budget reached - stopping training"); stop = True; break
    if stop:
        break
if not RESUME:
    model.save_pretrained(f"{OUT}/ce_model"); tok.save_pretrained(f"{OUT}/ce_model")
print(f"training done: {step:,} steps, {(time.time() - t0) / 60:.1f} min")

train rows 295,486 | positives 127,146


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 step 500/9,234 loss 0.4608 (2.8 min)
epoch 1 step 1,000/9,234 loss 0.2623 (5.7 min)
epoch 1 step 1,500/9,234 loss 0.2621 (8.6 min)
epoch 1 step 2,000/9,234 loss 0.2853 (11.4 min)
epoch 1 step 2,500/9,234 loss 0.1083 (14.3 min)
epoch 1 step 3,000/9,234 loss 0.2795 (17.2 min)
epoch 1 step 3,500/9,234 loss 0.1199 (20.0 min)
epoch 1 step 4,000/9,234 loss 0.1524 (22.9 min)
epoch 1 step 4,500/9,234 loss 0.0834 (25.8 min)
epoch 2 step 5,000/9,234 loss 0.0772 (28.6 min)
epoch 2 step 5,500/9,234 loss 0.1753 (31.5 min)
epoch 2 step 6,000/9,234 loss 0.1309 (34.4 min)
epoch 2 step 6,500/9,234 loss 0.1125 (37.2 min)
epoch 2 step 7,000/9,234 loss 0.0842 (40.1 min)
epoch 2 step 7,500/9,234 loss 0.1024 (42.9 min)
epoch 2 step 8,000/9,234 loss 0.0648 (45.8 min)
epoch 2 step 8,500/9,234 loss 0.0936 (48.7 min)
epoch 2 step 9,000/9,234 loss 0.0650 (51.5 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training done: 9,234 steps, 53.1 min


In [5]:
# ===== CELL 5: score the HOLD fold (for blend tuning) + report =====
hv = val[val.fold == HOLD].reset_index(drop=True)
if os.path.exists(f"{OUT}/ce_val_scores.parquet"):
    hv["ce"] = pd.read_parquet(f"{OUT}/ce_val_scores.parquet").set_index("row").ce.reindex(hv.row).values
else:
    hv["ce"] = score(model, hv)
hv[["row", "ce"]].to_parquet(f"{OUT}/ce_val_scores.parquet", index=False)
rep = {"hold_rows": len(hv), "train_rows": len(tr), "steps": step}
m = hv.band.values
for name, col in (("lgbm", "p"), ("ce", "ce")):
    x = np.clip(hv[col].values, 1e-6, 1 - 1e-6)
    rep[f"{name}_auc_band"] = float(roc_auc_score(hv.y[m], x[m])) if m.any() and hv.y[m].nunique() > 1 else None
    rep[f"{name}_logloss_band"] = float(log_loss(hv.y[m], x[m], labels=[0, 1])) if m.any() else None
print(json.dumps(rep, indent=1))
json.dump(rep, open(f"{OUT}/ce_report.json", "w"), indent=1)

  scored 204,800/466,440 (170s)
  scored 409,600/466,440 (407s)
{
 "hold_rows": 466440,
 "train_rows": 295486,
 "steps": 9234,
 "lgbm_auc_band": 0.9583764890982264,
 "lgbm_logloss_band": 0.23422865152681008,
 "ce_auc_band": 0.9718458114039018,
 "ce_logloss_band": 0.2107642302741442
}


In [6]:
# ===== CELL 6: score the contested test pairs =====
t0 = time.time()
if os.path.exists(f"{OUT}/ce_test_scores.parquet"):
    test["ce"] = pd.read_parquet(f"{OUT}/ce_test_scores.parquet").set_index("row").ce.reindex(test.row).values
else:
    test["ce"] = score(model, test)
test[["row", "ce"]].to_parquet(f"{OUT}/ce_test_scores.parquet", index=False)
print(f"test scored: {len(test):,} pairs in {(time.time() - t0) / 60:.1f} min")
print(test.groupby("country")["ce"].describe().to_string())

  scored 204,800/2,191,300 (110s)
  scored 409,600/2,191,300 (237s)
  scored 614,400/2,191,300 (379s)
  scored 819,200/2,191,300 (537s)
  scored 1,024,000/2,191,300 (709s)
  scored 1,228,800/2,191,300 (894s)
  scored 1,433,600/2,191,300 (1095s)
  scored 1,638,400/2,191,300 (1311s)
  scored 1,843,200/2,191,300 (1544s)
  scored 2,048,000/2,191,300 (1807s)
test scored: 2,191,300 pairs in 33.6 min
            count      mean       std       min       25%       50%       75%       max
country                                                                                
France   501540.0  0.413521  0.404419  0.000138  0.003248  0.275812  0.859428  0.999815
India    903474.0  0.254781  0.376926  0.000136  0.000175  0.012100  0.436142  0.999813
US       786286.0  0.218288  0.344381  0.000137  0.001053  0.040163  0.206094  0.999815


In [ ]:
# ===== CELL 8: unseen-country self-training (run after the notebook finished; ~60-90 min) =====
# Countries absent from training (test countries not in the validation pairs). Take their test pairs where the cross-encoder AND LightGBM agree confidently,
# fine-tune a COPY of the model on them (+ some original training pairs so it keeps what it learned), then rescore
# every unseen-country pair. Writes er_work/ce/ce_fr_test_scores.parquet (row, ce); the original files are untouched.
FR_OUT = f"{OUT}/ce_fr_test_scores.parquet"
if os.path.exists(FR_OUT):
    print("unseen-country scores already on Drive - skipping")
else:
    ce_now = pd.read_parquet(f"{OUT}/ce_test_scores.parquet").set_index("row").ce
    UNSEEN = set(test.country.astype(str)) - set(val.country.astype(str))
    fr = test[test.country.astype(str).isin(UNSEEN)].copy(); fr["ce"] = ce_now.reindex(fr.row).values
    pos = fr[(fr.ce >= 0.97) & (fr.p >= 0.97)]; neg = fr[(fr.ce <= 0.03) & (fr.p <= 0.03)]
    n = int(min(len(pos), len(neg), 100_000))
    assert n >= 5_000, f"only {n:,} confident unseen-country pairs per class - too few to fine-tune safely; skip this cell"
    pl = pd.concat([pos.sample(n, random_state=1).assign(y=1), neg.sample(n, random_state=1).assign(y=0)])
    keep_tr = tr.sample(min(len(tr), 2 * n), random_state=1)[["a", "b", "y"]]
    ft = pd.concat([pl[["a", "b", "y"]], keep_tr], ignore_index=True).sample(frac=1, random_state=1).reset_index(drop=True)
    print(f"unseen-country pseudo-labels: {n:,} positives + {n:,} negatives (from {len(fr):,} pairs) + {len(keep_tr):,} original")
    fm = AutoModelForSequenceClassification.from_pretrained(f"{OUT}/ce_model", num_labels=1).to(dev)
    opt2 = torch.optim.AdamW(fm.parameters(), lr=1e-5, weight_decay=0.01)
    st2 = math.ceil(len(ft) / BATCH); sch2 = get_linear_schedule_with_warmup(opt2, int(0.1 * st2), st2)
    sc2 = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
    a2, b2, y2 = ft.a.tolist(), ft.b.tolist(), ft.y.values.astype(np.float32)
    fm.train(); t0 = time.time()
    for k, s in enumerate(range(0, len(ft), BATCH)):
        rows = np.arange(s, min(s + BATCH, len(ft)))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            loss = lossf(fm(**encode(a2, b2, rows)).logits.squeeze(-1).float(), torch.tensor(y2[rows], device=dev))
        opt2.zero_grad(set_to_none=True); sc2.scale(loss).backward(); sc2.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(fm.parameters(), 1.0); sc2.step(opt2); sc2.update(); sch2.step()
        if (k + 1) % 500 == 0:
            print(f"  step {k + 1:,}/{st2:,} loss {loss.item():.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    fr_all = test[test.country.astype(str).isin(UNSEEN)].reset_index(drop=True)
    fr_all["ce"] = score(fm, fr_all)
    fr_all[["row", "ce"]].to_parquet(FR_OUT, index=False)
    old = ce_now.reindex(fr_all.row).values
    print(f"unseen-country pairs rescored: {len(fr_all):,} pairs | mean ce {old.mean():.3f} -> {fr_all.ce.mean():.3f} | "
          f"share >= 0.5: {(old >= 0.5).mean():.3f} -> {(fr_all.ce >= 0.5).mean():.3f} ({(time.time() - t0) / 60:.1f} min)")
    del fm; gc.collect(); torch.cuda.empty_cache()

Done - both score files are already on Drive in `er_work/ce/`. Next: the blend cell in `er_validation_v4.ipynb`.